# Yotuubef Colab GPU Runner

Updated notebook for running this repo on Google Colab with GPU and secret-based credentials.

## Before you run
1. In Colab: **Runtime -> Change runtime type -> GPU**
2. Add these in **Colab Secrets** (left sidebar key icon):
   - `REDDIT_CLIENT_ID`
   - `REDDIT_CLIENT_SECRET`
   - `REDDIT_USER_AGENT` (optional, default is set)
   - `NVIDIA_NIM_API_KEY`
   - `YOUTUBE_TOKEN_JSON` (required only when upload mode is enabled)


In [ ]:
# Setup system + clone repo + install dependencies
!nvidia-smi
!apt-get -qq update
!apt-get -qq install -y ffmpeg

REPO_URL = "https://github.com/beenycool/yotuubef.git"
REPO_REF = "main"
REPO_DIR = "/content/yotuubef"

import os

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 --branch $REPO_REF $REPO_URL $REPO_DIR

%cd /content/yotuubef
!git fetch origin $REPO_REF
!git checkout $REPO_REF
!git pull

!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements-ci.txt
!python -m pip install -q \
  google-generativeai google-api-python-client google-auth-oauthlib google-auth-httplib2 \
  imageio-ffmpeg scipy librosa soundfile \
  gputil pynvml aiosqlite


In [ ]:
# Load secrets from Colab Secrets into environment variables
import os

try:
    from google.colab import userdata

    def get_secret(name, default=""):
        try:
            value = userdata.get(name)
            return value if value is not None else default
        except Exception:
            return default
except Exception:
    def get_secret(name, default=""):
        return os.getenv(name, default)

os.environ["REDDIT_CLIENT_ID"] = get_secret("REDDIT_CLIENT_ID")
os.environ["REDDIT_CLIENT_SECRET"] = get_secret("REDDIT_CLIENT_SECRET")
os.environ["REDDIT_USER_AGENT"] = get_secret("REDDIT_USER_AGENT", "python:VideoBot:v2.0 (by /u/yourusername)")
os.environ["NVIDIA_NIM_API_KEY"] = get_secret("NVIDIA_NIM_API_KEY")
os.environ["AI_PROVIDER"] = get_secret("AI_PROVIDER", "nvidia_nim")

youtube_token_json = get_secret("YOUTUBE_TOKEN_JSON", "")
if youtube_token_json:
    os.environ["YOUTUBE_TOKEN_JSON"] = youtube_token_json

required = ["REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET", "NVIDIA_NIM_API_KEY"]
missing = [key for key in required if not os.environ.get(key)]
if missing:
    raise ValueError(f"Missing required Colab Secrets: {missing}")

print("Environment variables are set.")
print("YOUTUBE_TOKEN_JSON found:", bool(os.environ.get("YOUTUBE_TOKEN_JSON")))


In [ ]:
# Runtime options
MODE = "single"  # "single" or "find"
REDDIT_URL = "https://www.reddit.com/r/oddlysatisfying/comments/<post_id>/"  # set when MODE=single
MAX_VIDEOS = 1  # used when MODE=find
RUN_FULL_UPLOAD = False  # True: use main.py and upload to YouTube if token is available

if RUN_FULL_UPLOAD and not os.environ.get("YOUTUBE_TOKEN_JSON"):
    raise ValueError("RUN_FULL_UPLOAD=True requires YOUTUBE_TOKEN_JSON in Colab Secrets")

if MODE not in {"single", "find"}:
    raise ValueError("MODE must be 'single' or 'find'")

print("Mode:", MODE)
print("Upload mode:", RUN_FULL_UPLOAD)
if MODE == "single":
    print("Configured URL:", REDDIT_URL)
else:
    print("Configured MAX_VIDEOS:", MAX_VIDEOS)


In [ ]:
# Run pipeline
import asyncio
import json
import subprocess
from pathlib import Path

def ffmpeg_has_nvenc() -> bool:
    try:
        out = subprocess.check_output(["ffmpeg", "-hide_banner", "-encoders"], stderr=subprocess.STDOUT, text=True)
        return "h264_nvenc" in out
    except Exception:
        return False

if RUN_FULL_UPLOAD:
    import shlex

    if MODE == "single":
        cmd = f"python3 main.py single {shlex.quote(REDDIT_URL)}"
    else:
        cmd = f"python3 main.py find --max-videos {int(MAX_VIDEOS)}"

    print("Running:", cmd)
    proc = subprocess.run(cmd, shell=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}")

else:
    if MODE != "single":
        raise ValueError("RUN_FULL_UPLOAD=False currently supports only MODE='single'")

    from src.config.settings import get_config
    from src.enhanced_orchestrator import EnhancedVideoOrchestrator

    cfg = get_config()
    if not ffmpeg_has_nvenc():
        cfg.video.enable_speed_optimization = False
        cfg.video.video_codec_cpu = "libx264"
        cfg.video.ffmpeg_cpu_preset = "ultrafast"
        print("NVENC not available: using CPU x264 fallback.")
    else:
        print("NVENC available: GPU encoding can be used.")

    async def run_local_pipeline(reddit_url: str):
        orchestrator = EnhancedVideoOrchestrator()

        download_result = await orchestrator._download_and_analyze_video(reddit_url)
        if not download_result.get("success"):
            return download_result

        video_path = download_result["video_path"]
        analysis = download_result["analysis"]

        analysis = await orchestrator._perform_enhanced_analysis(video_path, analysis)
        if orchestrator.enable_cinematic_editing:
            analysis = await orchestrator._apply_cinematic_analysis(video_path, analysis)

        process_result = await orchestrator._process_with_enhancements(video_path, analysis, options={})
        if not process_result.get("success"):
            return {"success": False, "stage": "process", "details": process_result}

        thumbnail_results = await orchestrator._generate_ab_test_thumbnails(video_path, analysis, process_result)

        result = {
            "success": True,
            "source_video": str(video_path),
            "processed_video": str(process_result.get("video_path")),
            "processing_stats": process_result.get("processing_stats", {}),
            "thumbnail_variants": thumbnail_results,
        }

        out_dir = Path("data/results")
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / "colab_run_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        return result

    local_result = asyncio.run(run_local_pipeline(REDDIT_URL))
    print(json.dumps(local_result, indent=2))


In [ ]:
# Show latest result file (if any)
from pathlib import Path

results_dir = Path("data/results")
if results_dir.exists():
    files = sorted(results_dir.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if files:
        print("Latest result:", files[0])
        print(files[0].read_text(encoding="utf-8")[:6000])
    else:
        print("No JSON result files found yet.")
else:
    print("Results directory does not exist yet.")


In [ ]:
# Optional: download latest processed video from Colab runtime
from pathlib import Path

try:
    from google.colab import files
except Exception:
    files = None

processed_dir = Path("processed")
if processed_dir.exists():
    videos = sorted(processed_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)
    if videos:
        print("Latest video:", videos[0])
        if files is not None:
            files.download(str(videos[0]))
        else:
            print("Not running in Colab; download skipped.")
    else:
        print("No processed .mp4 files found.")
else:
    print("Processed directory does not exist yet.")
